# PyTorch Tensors Deep Dive: dtypes, Device Movement, Memory Layout, and Broadcasting

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/pytorch_1)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
import torch

# Default dtype is float32
x = torch.tensor([1.0, 2.0, 3.0])
print(f"dtype: {x.dtype}")
print(f"shape: {x.shape}")
print(f"device: {x.device}")

In [ ]:
import torch

# Python int literal → torch.int64 by default
a = torch.tensor([1, 2, 3])

# Python float literal → torch.float32 by default
b = torch.tensor([1.0, 2.0, 3.0])

# Explicit dtype
c = torch.tensor([1, 2, 3], dtype=torch.float32)
d = torch.tensor([1, 2, 3], dtype=torch.int8)

print(a.dtype, b.dtype, c.dtype, d.dtype)

In [ ]:
import torch

x = torch.tensor([1, 2, 3], dtype=torch.int64)

# Preferred: .to()
x_float = x.to(torch.float32)

# Shorthand (equivalent)
x_float2 = x.float()  # → float32
x_half = x.half()     # → float16

print(x_float.dtype, x_float2.dtype, x_half.dtype)

In [ ]:
import torch

a = torch.tensor([1.0], dtype=torch.float16)
b = torch.tensor([2.0], dtype=torch.float32)

result = a + b
print(result.dtype)  # promoted to float32

In [ ]:
import torch

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

x = torch.tensor([1.0, 2.0, 3.0])
print(f"x device: {x.device}")

# Move to GPU
x_gpu = x.to(device)
print(f"x_gpu device: {x_gpu.device}")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Slow: CPU allocation + copy
x_slow = torch.zeros(1000, 1000).to(device)

# Fast: direct allocation on target device
x_fast = torch.zeros(1000, 1000, device=device)

print(x_fast.device)

In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.tensor([1.0, 2.0, 3.0], device=device, requires_grad=True)

# This would fail on GPU:
# arr = x.numpy()  # RuntimeError

# Correct pattern:
arr = x.detach().cpu().numpy()
print(arr, type(arr))

In [ ]:
import torch

x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]], dtype=torch.float32)

print(f"shape:   {x.shape}")
print(f"strides: {x.stride()}")
print(f"is_contiguous: {x.is_contiguous()}")

In [ ]:
import torch

x = torch.randn(3, 4)
print(f"x strides: {x.stride()}, contiguous: {x.is_contiguous()}")

xt = x.T  # or x.transpose(0, 1)
print(f"xt strides: {xt.stride()}, contiguous: {xt.is_contiguous()}")

In [ ]:
import torch

x = torch.randn(3, 4).T  # non-contiguous

try:
    x.view(12)  # view() requires contiguous
except RuntimeError as e:
    print(f"Error: {e}")

# Fix: call .contiguous() first
x_c = x.contiguous()
print(x_c.view(12).shape)

In [ ]:
import torch

a = torch.tensor([[1.0], [2.0], [3.0]])  # shape (3, 1)
b = torch.tensor([10.0, 20.0, 30.0])    # shape (3,) → (1, 3) after prepend

result = a + b  # broadcasts to (3, 3)
print(f"a shape: {a.shape}")
print(f"b shape: {b.shape}")
print(f"result shape: {result.shape}")
print(result)

In [ ]:
import torch

# batch of 4 samples, 3 features each
batch = torch.randn(4, 3)

mean = batch.mean(dim=0)   # shape (3,)
std  = batch.std(dim=0)    # shape (3,)

# mean/std broadcast over batch dimension automatically
normalized = (batch - mean) / (std + 1e-8)

print(f"batch:      {batch.shape}")
print(f"mean:       {mean.shape}")
print(f"normalized: {normalized.shape}")
print(f"col means after norm: {normalized.mean(dim=0).round(decimals=4)}")

In [ ]:
import torch

# Intended: add a bias to each of 4 samples
bias = torch.tensor([1.0, 2.0, 3.0])   # shape (3,) — correct
wrong_bias = torch.tensor([[1.0], [2.0], [3.0]])  # shape (3,1) — wrong!

x = torch.zeros(4, 3)

correct = x + bias        # (4,3) + (3,) → (4,3) ✓
wrong   = x + wrong_bias  # (4,3) + (3,1) → (4,3) but adds wrong values!

print(f"correct shape: {correct.shape}")
print(f"wrong shape:   {wrong.shape}")
print(f"correct[0]:    {correct[0]}")
print(f"wrong[0]:      {wrong[0]}")